In [8]:
from pathlib import Path # baseline variables that can be used later
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

In [14]:
import pandas as pd # see how many fights i got and what the total results were in all of them

df = pd.read_csv(f"{RAW}/ufc-master.csv")
df["date"] = pd.to_datetime(df["date"])
print(df.shape, df["date"].min(), df["date"].max())
print(df["Winner"].value_counts(dropna=False))

(7177, 118) 2010-03-21 00:00:00 2026-03-28 00:00:00
Winner
Red           4145
Blue          3024
Draw             5
No Contest       3
Name: count, dtype: int64


In [10]:
# keep only decisive fights
df = df[df["Winner"].isin(["Red", "Blue"])].copy()
df["target"] = (df["Winner"] == "Red").astype(int)
df = df.sort_values("date").reset_index(drop=True)
print(df.shape, df["target"].mean())

# leakage columns: they describe what happened in the fight that the machine needs to predict
leak = ["finish", "finish_details", "finish_round",
        "finish_round_time", "total_fight_time_secs"]

# how much data is missing, by column
missing = df.isna().mean().sort_values(ascending=False)
print(missing[missing > 0].head(40))

# fights per year, total is 7169
print(df["date"].dt.year.value_counts().sort_index())

(7169, 119) 0.5781838471195425
B_Women's Featherweight_rank    0.999861
R_Women's Featherweight_rank    0.998745
B_Pound-for-Pound_rank          0.988701
B_Women's Flyweight_rank        0.988143
R_Women's Flyweight_rank        0.984517
B_Women's Strawweight_rank      0.984098
B_Women's Bantamweight_rank     0.983261
B_Welterweight_rank             0.981727
B_Light Heavyweight_rank        0.981587
B_Featherweight_rank            0.981308
B_Lightweight_rank              0.981169
B_Bantamweight_rank             0.981169
B_Flyweight_rank                0.980053
B_Middleweight_rank             0.978937
B_Heavyweight_rank              0.977961
R_Women's Strawweight_rank      0.977124
R_Women's Bantamweight_rank     0.976566
R_Featherweight_rank            0.972381
R_Light Heavyweight_rank        0.971823
R_Lightweight_rank              0.971823
R_Heavyweight_rank              0.971823
R_Bantamweight_rank             0.971684
R_Middleweight_rank             0.971684
R_Welterweight_rank       

In [11]:
import numpy as np

cols = ["R_odds", "B_odds", "r_ko_odds", "R_avg_SIG_STR_landed",
        "R_match_weightclass_rank"]
df["year"] = df["date"].dt.year
print(df.groupby("year")[cols].apply(lambda g: g.isna().mean()).round(2))

# baseline: pick the favorite from the moneyline odds
def implied(o):
    return np.where(o < 0, -o / (-o + 100), 100 / (o + 100))

df["r_prob"] = implied(df["R_odds"])
df["b_prob"] = implied(df["B_odds"])
has_odds = df[["R_odds", "B_odds"]].notna().all(axis=1)

pick_red = (df["r_prob"] > df["b_prob"]).astype(int)
correct = (pick_red == df["target"])

print("fights with odds:", has_odds.sum(), "of", len(df))
print("favorite accuracy:", correct[has_odds].mean())
print("always-Red accuracy:", df["target"].mean())
print(correct[has_odds].groupby(df["year"]).mean().round(3))

      R_odds  B_odds  r_ko_odds  R_avg_SIG_STR_landed  \
year                                                    
2010    0.00    0.00       1.00                  0.10   
2011    0.00    0.00       1.00                  0.19   
2012    0.00    0.00       0.78                  0.10   
2013    0.00    0.00       0.11                  0.11   
2014    0.00    0.00       0.35                  0.14   
2015    0.00    0.00       0.02                  0.06   
2016    0.00    0.00       0.04                  0.04   
2017    0.00    0.00       0.02                  0.09   
2018    0.00    0.00       0.03                  0.07   
2019    0.00    0.00       0.01                  0.18   
2020    0.00    0.00       0.01                  0.05   
2021    0.00    0.00       0.02                  0.00   
2022    0.03    0.03       0.07                  0.00   
2023    0.16    0.17       0.19                  0.00   
2024    0.25    0.25       0.30                  0.00   
2025    0.03    0.02       0.05

In [12]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# 1. Load
df = pd.read_csv(RAW / "ufc-master.csv")
df["date"] = pd.to_datetime(df["date"])

# 2. Keep only fights with a winner, make the target, sort by date
df = df[df["Winner"].isin(["Red", "Blue"])].copy()
df["target"] = (df["Winner"] == "Red").astype(int)   # 1 = Red won
df = df.sort_values("date").reset_index(drop=True)
df["fight_id"] = df.index                            # used later to merge history features

# 3. Drop the columns we decided are useless
rank_cols = [c for c in df.columns if "rank" in c.lower()]
method_odds = ["r_dec_odds", "b_dec_odds", "r_sub_odds",
               "b_sub_odds", "r_ko_odds", "b_ko_odds"]
df = df.drop(columns=rank_cols + method_odds + ["empty_arena"])

# 4. Result columns: kept in df (needed for history features), left out of df_clean
result_cols = ["finish", "finish_details", "finish_round",
               "finish_round_time", "total_fight_time_secs"]
df_clean = df.drop(columns=result_cols)

# 5. Save both versions
df.to_csv(PROCESSED / "ufc_with_results.csv", index=False)
df_clean.to_csv(PROCESSED / "ufc_clean.csv", index=False)

# 6. Quick checks
print("df:", df.shape, "| df_clean:", df_clean.shape)
print(df["date"].min(), "to", df["date"].max())
print("Red win rate:", df["target"].mean().round(3))
print(df["finish"].value_counts(dropna=False))
print(df["finish_details"].value_counts().head(30))

df: (7169, 84) | df_clean: (7169, 79)
2010-03-21 00:00:00 to 2026-03-28 00:00:00
Red win rate: 0.578
finish
U-DEC         2680
KO/TKO        2223
SUB           1260
S-DEC          699
NaN            238
M-DEC           49
DQ              18
Overturned       2
Name: count, dtype: int64
finish_details
Punches                765
Punch                  744
Rear Naked Choke       509
Guillotine Choke       215
Kick                   164
Armbar                 126
Elbows                 107
Arm Triangle           104
Knee                    94
Triangle Choke          57
D'Arce Choke            41
Elbow                   37
Kimura                  30
Anaconda Choke          28
Flying Knee             26
Knees                   23
Kneebar                 17
Heel Hook               15
Other - Choke           14
Neck Crank              13
Spinning Back Fist      10
Slam                     9
Spinning Back Kick       9
Spinning Back Elbow      7
Other - Lock             6
Triangle Armbar         